# VisoSwap — Free GPU Runner on Kaggle

Run VisoSwap Studio on Kaggle's free GPU (NVIDIA T4 or P100).

### Instructions:
1. In the right panel under **Settings**:
   - **Accelerator**: Select **GPU T4 x2** (or **GPU P100**)
   - **Internet**: Toggle to **Internet on** (required for downloads and Cloudflare tunnel)
2. Click **Run All** (or run cells one-by-one).
3. Wait for the public Cloudflare tunnel URL printed at the bottom cell (e.g. `https://xxxx.trycloudflare.com`).
4. Open the link on any browser or mobile device (Safari/Chrome) to use VisoSwap with live stream swap and theater mode!

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

In [ ]:
# Step 2: Install System Dependencies & Cloudflare Tunnel
!apt-get update -qq && apt-get install -y -qq ffmpeg curl
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Step 3: Clone VisoSwap Repository (including pre-built Web UI)
import os
if not os.path.exists('/kaggle/working/visoswap'):
    !git clone --depth 1 https://github.com/kaiser62/visoswap.git /kaggle/working/visoswap
else:
    %cd /kaggle/working/visoswap
    !git pull origin master
%cd /kaggle/working/visoswap
!ls -la frontend/dist

In [ ]:
# Step 4: Install Python & Engine Dependencies with Clean GPU ONNX Runtime
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install --no-cache-dir onnxruntime-gpu

!pip install --no-cache-dir \
    fastapi uvicorn[standard] pydantic pydantic-settings aiosqlite httpx \
    python-multipart yt-dlp opencv-python Pillow ftfy regex numexpr onnxsim requests tqdm

# Check CUDA execution provider is available
import onnxruntime as ort
print('Available providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider not detected!'

In [ ]:
# Step 5: Choose Model Download Option
# - 'default': Download only 6 models needed for default face swapping (~1.3 GB, FAST ~30 sec)
# - 'all': Download all 56 models (~12 GB, complete offline catalog)
MODEL_SUBSET = 'default'  # Change to 'all' if you want every optional model/enhancer

import os
from pathlib import Path

models_dir = Path('/kaggle/working/visoswap/model_assets_owned')
models_dir.mkdir(parents=True, exist_ok=True)
os.environ['MODELS_DIR'] = str(models_dir)
os.environ['MODELS_SUBSET'] = MODEL_SUBSET

from visoswap.models.bootstrap import repair, verify, FAST
from visoswap.models import manifest

print(f'Starting model bootstrap (subset={MODEL_SUBSET})...')
res = repair(models_dir=models_dir, mode=FAST, subset=MODEL_SUBSET)
print(f'Bootstrap status: {"OK" if res.ok else "INCOMPLETE"}')
print(f'Required models checked: {res.required_checked}, Present: {len(res.present)}')
for entry in res.present:
    print(f'  ✓ {entry.name} ({entry.path.name})')
if not res.ok:
    print('Missing entries:', [e.name for e in res.absent])
    raise RuntimeError('Model download incomplete!')

In [ ]:
# Step 6: Launch Cloudflare Tunnel and Start VisoSwap Backend Server
import subprocess
import time
import re
import os

os.environ['MODELS_DIR'] = '/kaggle/working/visoswap/model_assets_owned'
os.environ['MODELS_VERIFY_MODE'] = 'fast'
os.environ['MODELS_SUBSET'] = MODEL_SUBSET

# 1. Start Cloudflare Tunnel in background
tunnel_log = open('/kaggle/working/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# 2. Extract public trycloudflare.com URL
print('Waiting for Cloudflare Tunnel to connect...')
public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/kaggle/working/tunnel.log'):
        content = open('/kaggle/working/tunnel.log').read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('=' * 65)
    print('🚀 VisoSwap is LIVE! Access your Web UI at:')
    print(f'👉 {public_url}')
    print(f'👉 Mobile UI: {public_url}/mobile')
    print('=' * 65)
else:
    print('Could not find tunnel URL yet. Check /kaggle/working/tunnel.log')

# 3. Run FastAPI/Uvicorn server
!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000
